# 02 · Part 2: HNSW, IVF-Flat and IVF-PQ

On both datasets, each index is built with a few settings, and each build's search parameter (efSearch or nprobe) is swept from low recall upwards. A sweep stops once the tune-half recall reaches `recall_stop`, or at the end of the grid.

Every setting is timed one query at a time on one pinned thread (median of `repeats` passes), and scored with our recall@10 on both query halves. Configurations are chosen on the tune half; the numbers reported are from the report half. The exact baseline comes from Part 1, timed with the same protocol on the same base set. Measurement choices are listed in `design_note.md`.

In [ ]:
%run ./00_core.ipynb

## Grids
The grids for this mode come from `CONFIG`. IVF-PQ uses 8-bit codes (FAISS's default), so m sub-vectors take m bytes per vector, and m must divide the dimension. nprobe values larger than nlist are skipped.

In [ ]:
def part2_configs(name, config=CONFIG):
    """(index kind, build settings, search values) for every Part 2 build on dataset `name`."""
    probes = lambda nlist: [p for p in config["nprobe"] if p <= nlist]
    return ([("hnsw", dict(M=M, efC=config["hnsw_efC"]), config["ef_search"]) for M in config["hnsw_M"]]
            + [("ivf_flat", dict(nlist=nlist), probes(nlist)) for nlist in config["ivf_nlist"]]
            + [("ivf_pq", dict(nlist=config["ivfpq_nlist"], m=m), probes(config["ivfpq_nlist"]))
               for m in config["ivfpq_m"][name]])


pd.DataFrame([dict(dataset=name, index=kind, build=json.dumps(params), search=f"{SEARCH_PARAM[kind]} in {values}")
              for name in DATASETS for kind, params, values in part2_configs(name)])

In [ ]:
%%ipytest -qq

@pytest.mark.parametrize("name", ["sift", "glove"])
def test_full_grid_tries_several_build_settings_per_index(name):
    kinds = [kind for kind, _, _ in part2_configs(name, FULL)]
    assert all(kinds.count(kind) >= 2 for kind in ("hnsw", "ivf_flat", "ivf_pq"))


@pytest.mark.parametrize("config", [FULL, QUICK], ids=["full", "quick"])
def test_every_index_runs_and_nprobe_never_exceeds_nlist(config):
    for name in DATASETS:
        runs = part2_configs(name, config)
        assert {kind for kind, _, _ in runs} == {"hnsw", "ivf_flat", "ivf_pq"}
        assert all(max(values) <= params["nlist"] for kind, params, values in runs if kind != "hnsw")

## Run
Each dataset's rows are written to `results.csv` as soon as its runs finish, so a failure on the second dataset keeps the first.

In [ ]:
for name in DATASETS:
    started = time.perf_counter()
    data = load_dataset(name, CONFIG["n_base"])
    gt = ground_truth(name, data.base, data.queries)
    rows = []
    for kind, params, values in part2_configs(name):
        rows += benchmark_index(2, name, data, gt, kind, params, values)
    write_rows(rows, part=2, section=name)
    print(f"{name}: {len(rows)} rows in {(time.perf_counter() - started) / 60:.1f} min")
    del data, gt

## Recall against speed
Each line is an index's best trade-off curve: the tune-half Pareto front over all its build settings and search values, drawn at its report-half recall and QPS. Faint dots are every configuration measured (report half). The grey line and diamond are exact search from Part 1. The vertical lines mark recall 0.90 and 0.95.

In [ ]:
results = read_results()
figure, tables = part2_assets(results)
plt.show()

## Tables
- **QPS at recall 0.90 and 0.95:** the fastest configuration whose tune-half recall reaches the target, with its report-half recall and QPS.
- **Cost at recall 0.95:** that configuration's build time, serialised size and bytes per vector, plus the RAM IVF-PQ keeps for its precomputed search table. An index that never reaches 0.95 is shown at its highest-recall configuration and marked "no".
- **Highest recall reached:** for every build, the best tune-half recall in its sweep. This is the evidence for any index that fails to reach high recall.

In [ ]:
for name, table in tables.items():
    print(name)
    display(table)

In [ ]:
%%ipytest -qq

part2 = results[results["part"] == 2]


def test_every_setting_is_measured_on_both_halves():
    splits = part2.groupby(["dataset", "index", "build_params", "param_value"])["split"].agg(frozenset)
    assert (splits == frozenset({"tune", "report"})).all()


def test_hnsw_and_ivf_flat_reach_the_top_of_the_recall_range():
    best = part2[part2["split"] == "tune"].groupby(["dataset", "index"])["recall_at_10"].max().unstack()
    assert (best[["hnsw", "ivf_flat"]] >= 0.95).all().all()


def test_tables_cover_every_index_and_exact_search_on_both_datasets():
    for table in (qps_at_recall_table(results), cost_at_recall_table(results)):
        assert len(table) == len(DATASETS) * 4
    assert (FIGURES_DIR / "part2_recall_qps.pdf").exists() and (TABLES_DIR / "part2_qps_at_recall.tex").exists()